In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Perceptron
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import joblib

In [2]:
# ==========================================
# 1. Carregamento e Limpeza Inicial de Dados
# ==========================================
print("Carregando dados...")
url = "https://www.fhwa.dot.gov/bridge/nbi/2025/delimited/NY25.txt"

df = pd.read_csv(url, sep=",", quotechar="'", low_memory=False)
df_oae = df.copy()
print(f"Shape antes de limpeza inicial: {df_oae.shape}")

Carregando dados...
Shape antes de limpeza inicial: (17666, 123)


In [3]:
# Remover colunas com mais de 50% de valores ausentes
missing_percentage = df_oae.isnull().sum() / len(df_oae)
cols_to_drop = missing_percentage[missing_percentage > 0.5].index
df_oae.drop(columns=cols_to_drop, inplace=True)

In [4]:
# Remover duplicatas e linhas onde o alvo primário é nulo
df_oae.drop_duplicates(inplace=True)
df_oae.dropna(subset=['BRIDGE_CONDITION'], inplace=True)

print(f"Shape após limpeza inicial: {df_oae.shape}")

Shape após limpeza inicial: (17666, 112)


In [5]:
# ==========================================
# 2. Identificação e Remoção de Data Leakage
# ==========================================
leaky_features = [
    'DECK_COND_058', 'SUPERSTRUCTURE_COND_059', 'SUBSTRUCTURE_COND_060',
    'CHANNEL_COND_061', 'CULVERT_COND_062', 'STRUCTURAL_EVAL_067',
    'DECK_GEOMETRY_EVAL_068', 'UNDCLRENCE_EVAL_069', 'POSTING_EVAL_070',
    'WATERWAY_EVAL_071', 'APPR_ROAD_EVAL_072', 'OPERATING_RATING_064',
    'INVENTORY_RATING_066', 'LOWEST_RATING'
]

leaky_features_present = [col for col in leaky_features if col in df_oae.columns]
df_sem_leakage = df_oae.drop(columns=leaky_features_present, errors='ignore')
print(f"Features de condição (leakage) removidas: {len(leaky_features_present)}")

Features de condição (leakage) removidas: 14


In [6]:
# ==========================================
# 3. Seleção de Features (Análise de Domínio)
# ==========================================
features_nao_condicionais = [
    'YEAR_BUILT_027', 'YEAR_RECONSTRUCTED_106', 'STRUCTURE_KIND_043A',
    'STRUCTURE_TYPE_043B', 'APPR_KIND_044A', 'APPR_TYPE_044B',
    'MAIN_UNIT_SPANS_045', 'APPR_SPANS_046', 'MAX_SPAN_LEN_MT_048',
    'STRUCTURE_LEN_MT_049', 'DECK_STRUCTURE_TYPE_107', 'SURFACE_TYPE_108A',
    'MEMBRANE_TYPE_108B', 'DECK_PROTECTION_108C', 'DESIGN_LOAD_031',
    'DEGREES_SKEW_034', 'STRUCTURE_FLARED_035', 'BRIDGE_LEN_IND_112',
    'DECK_AREA', 'PARALLEL_STRUCTURE_101', 'TEMP_STRUCTURE_103',
    'APPR_WIDTH_MT_032', 'MEDIAN_CODE_033', 'HORR_CLR_MT_047',
    'LEFT_CURB_MT_050A', 'RIGHT_CURB_MT_050B', 'ROADWAY_WIDTH_MT_051',
    'DECK_WIDTH_MT_052', 'MIN_VERT_CLR_010', 'VERT_CLR_OVER_MT_053',
    'VERT_CLR_UND_054B', 'LAT_UND_MT_055B', 'LEFT_LAT_UND_MT_056',
    'ADT_029', 'YEAR_ADT_030', 'PERCENT_ADT_TRUCK_109', 'FUTURE_ADT_114',
    'YEAR_OF_FUTURE_ADT_115', 'TRAFFIC_LANES_ON_028A', 'TRAFFIC_LANES_UND_028B',
    'ROUTE_PREFIX_005B', 'SERVICE_LEVEL_005C', 'ROUTE_NUMBER_005D',
    'DIRECTION_005E', 'FUNCTIONAL_CLASS_026', 'BASE_HWY_NETWORK_012',
    'HIGHWAY_SYSTEM_104', 'NATIONAL_NETWORK_110', 'STRAHNET_HIGHWAY_100',
    'TRAFFIC_DIRECTION_102', 'SERVICE_ON_042A', 'SERVICE_UND_042B',
    'DETOUR_KILOS_019', 'OWNER_022', 'MAINTENANCE_021', 'TOLL_020',
    'HISTORY_037', 'FEDERAL_LANDS_105', 'RAILINGS_036A', 'TRANSITIONS_036B',
    'APPR_RAIL_036C', 'APPR_RAIL_END_036D', 'BRIDGE_CONDITION'
]

features_existentes = [col for col in features_nao_condicionais if col in df_sem_leakage.columns]
df_reduzido = df_sem_leakage[features_existentes].copy()

In [7]:
# ==========================================
# 4. Engenharia de Features e Preparação do Alvo
# ==========================================
# Idade e Densidade
current_year = datetime.now().year
df_reduzido['AGE'] = current_year - df_reduzido['YEAR_BUILT_027']
df_reduzido.loc[df_reduzido['AGE'] < 0, 'AGE'] = 0

if 'ADT_029' in df_reduzido.columns and 'TRAFFIC_LANES_ON_028A' in df_reduzido.columns:
    df_reduzido['TRAFFIC_DENSITY'] = df_reduzido['ADT_029'] / (df_reduzido['TRAFFIC_LANES_ON_028A'] + 1)
    df_reduzido['TRAFFIC_DENSITY'] = df_reduzido['TRAFFIC_DENSITY'].replace([np.inf, -np.inf], np.nan)

df_reduzido['AGE_NORMALIZED'] = (df_reduzido['AGE'] - df_reduzido['AGE'].mean()) / df_reduzido['AGE'].std()

In [8]:
# Transformando a variável alvo (1 = Crítico/Fair/Poor, 0 = Good)
map_bridge_condition = {'G': 0, 'F': 1, 'P': 1}
df_reduzido['TARGET'] = df_reduzido['BRIDGE_CONDITION'].map(map_bridge_condition)

# CRÍTICO: Remover valores que não mapearam (ex: 'N' ou vazios) e converter para inteiro
df_reduzido.dropna(subset=['TARGET'], inplace=True)
df_reduzido['TARGET'] = df_reduzido['TARGET'].astype(int)
df_reduzido.drop(columns=['BRIDGE_CONDITION'], inplace=True)

In [9]:
# ==========================================
# 5. Pipeline de Pré-processamento e Divisão
# ==========================================
X = df_reduzido.drop(columns=['TARGET'])
y = df_reduzido['TARGET']

num_cols = X.select_dtypes(include=['int64', 'float64']).columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols),
        ('cat', categorical_transformer, cat_cols)
    ])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [10]:
# ==========================================
# 6. Modelagem: Baseline Perceptron
# ==========================================
pipeline_perceptron = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', Perceptron(max_iter=1000, tol=1e-3, random_state=42))
])

pipeline_perceptron.fit(X_train, y_train)
y_pred_perc = pipeline_perceptron.predict(X_test)

print("\n--- Métricas Perceptron Baseline ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_perc):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_perc, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_perc):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_perc):.4f}")


--- Métricas Perceptron Baseline ---
Accuracy:  0.7140
Precision: 0.7693
Recall:    0.8254
F1-Score:  0.7963


In [11]:
# ==========================================
# 7. Modelagem: Árvore de Decisão (Default e Otimizada)
# ==========================================
pipeline_dt = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

pipeline_dt.fit(X_train, y_train)
y_pred_dt = pipeline_dt.predict(X_test)

print("\n--- Métricas Árvore de Decisão (Default) ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dt):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_dt):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_dt):.4f}")

param_grid_dt = {
    'classifier__max_depth': [5, 10, 15, None],
    'classifier__min_samples_split': [2, 10, 20],
    'classifier__min_samples_leaf': [1, 5, 10],
    'classifier__class_weight': ['balanced']
}

print("\nOtimizando Árvore de Decisão...")
grid_search_dt = GridSearchCV(pipeline_dt, param_grid_dt, cv=5, scoring='f1', n_jobs=-1)
grid_search_dt.fit(X_train, y_train)

y_pred_dt_opt = grid_search_dt.predict(X_test)
print(f"Melhores parâmetros (DT): {grid_search_dt.best_params_}")
print(f"F1-Score (DT Otimizada): {f1_score(y_test, y_pred_dt_opt):.4f}")


--- Métricas Árvore de Decisão (Default) ---
Accuracy:  0.7557
Precision: 0.8227
Recall:    0.8151
F1-Score:  0.8189

Otimizando Árvore de Decisão...
Melhores parâmetros (DT): {'classifier__class_weight': 'balanced', 'classifier__max_depth': 5, 'classifier__min_samples_leaf': 1, 'classifier__min_samples_split': 20}
F1-Score (DT Otimizada): 0.8405


In [12]:
# ==========================================
# 8. Modelagem: Random Forest (Ensemble)
# ==========================================
pipeline_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(class_weight='balanced', random_state=42))
])

param_grid_rf = {
    'classifier__n_estimators': [100, 200],
    'classifier__max_depth': [10, 20, None],
}

print("\nOtimizando Random Forest...")
grid_search_rf = GridSearchCV(pipeline_rf, param_grid_rf, cv=5, scoring='f1', n_jobs=-1)
grid_search_rf.fit(X_train, y_train)

y_pred_rf = grid_search_rf.predict(X_test)

print("\n--- Métricas Random Forest Otimizada ---")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_rf):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred_rf):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred_rf):.4f}")


Otimizando Random Forest...

--- Métricas Random Forest Otimizada ---
Accuracy:  0.8174
Precision: 0.8386
Recall:    0.9045
F1-Score:  0.8703


In [13]:
# ==========================================
# 9. Deploy / Exportação dos Modelos
# ==========================================
print("\nSalvando modelos...")
# Salva o histórico completo (pesquisa/auditoria)
joblib.dump(grid_search_rf, 'modelo_rf_oae_completo.pkl')

# Salva apenas o melhor pipeline (produção/deploy web)
joblib.dump(grid_search_rf.best_estimator_, 'modelo_pontes_ny_rf_deploy.joblib')

print("Processo concluído com sucesso!")


Salvando modelos...
Processo concluído com sucesso!


In [ ]:
# ==========================================
# 10. Carregando e Testando o Modelo para Deploy
# ==========================================

import joblib
import pandas as pd

# Carregar o modelo de deploy
modelo_deploy = joblib.load('modelo_pontes_ny_rf_deploy.joblib')

print("Modelo carregado com sucesso!")

FileNotFoundError: [Errno 2] No such file or directory: '/content/modelo_pontes_ny_rf_deploy.joblib'

Vamos usar uma pequena amostra do `X_test` para fazer uma previsão e ver como o modelo se comporta.

In [ ]:
# Pegar uma amostra do X_test para demonstração
sample_for_prediction = X_test.head(5)

print("Amostra dos dados para previsão:")
display(sample_for_prediction)

# Fazer a previsão
predictions = modelo_deploy.predict(sample_for_prediction)

print("\nPrevisões para a amostra:")
print(predictions)

# Opcional: Se o modelo suportar, podemos obter as probabilidades
if hasattr(modelo_deploy, 'predict_proba'):
    probabilities = modelo_deploy.predict_proba(sample_for_prediction)
    print("\nProbabilidades de previsão para a amostra (Classe 0, Classe 1):")
    print(probabilities)


Amostra dos dados para previsão:


,YEAR_BUILT_027,YEAR_RECONSTRUCTED_106,STRUCTURE_KIND_043A,STRUCTURE_TYPE_043B,APPR_KIND_044A,APPR_TYPE_044B,MAIN_UNIT_SPANS_045,APPR_SPANS_046,MAX_SPAN_LEN_MT_048,STRUCTURE_LEN_MT_049,...,TOLL_020,HISTORY_037,FEDERAL_LANDS_105,RAILINGS_036A,TRANSITIONS_036B,APPR_RAIL_036C,APPR_RAIL_END_036D,AGE,TRAFFIC_DENSITY,AGE_NORMALIZED
6615,1937,0.0,1,1,0,0,1,0,6.7,7.0,...,3,5,0,1,0,1,1,89,1439.666667,1.179736
11511,1951,2015.0,5,5,0,0,1,0,7.9,8.5,...,3,5,0,1,0,1,0,75,494.666667,0.708251
15241,1952,0.0,3,2,0,0,1,0,9.1,9.4,...,3,5,0,1,N,0,0,74,39.000000,0.674574
13760,1991,0.0,9,19,0,0,2,0,5.1,12.5,...,3,5,0,0,0,1,1,35,91.666667,-0.638849
2019,2008,0.0,1,19,0,0,1,0,8.5,8.8,...,3,5,0,1,1,1,1,18,6185.000000,-1.211366



Previsões para a amostra:
[1 1 1 1 0]

Probabilidades de previsão para a amostra (Classe 0, Classe 1):
[[0.03  0.97 ]
 [0.255 0.745]
 [0.03  0.97 ]
 [0.295 0.705]
 [0.865 0.135]]
